# Competition Analysis - AI Actors Network

This notebook builds a competitive map of AI companies from `database.db`.

## Goal

Map competitive proximity from `company -> competitor` relations, then project that space into 2D.

## Pipeline

1. Extract companies with competitors and financial score.
2. Clean and normalize competitor names.
3. Build directed pairs `company -> competitor`.
4. Aggregate and build the directed matrix `company x competitors`.
5. Compute projections and generate interactive maps.

## Reading Convention

Each section follows: **Goal -> Inputs -> Processing -> Outputs**.

## Business Rules

- Population analyzed: `ranking_score > 0`.
- `ranking_score = capitalization`, or `funds_raised` when capitalization is missing.
- Map label size: `sqrt` mode (compact range 8-18).

## 1. Setup and Imports

**Goal**: load dependencies and define shared constants.

**Inputs**: project path.
**Outputs**: global variables (`DB_PATH`, `EXPORTS_DIR`, column names, seed).

In [1]:
import sqlite3
import re
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.manifold import TSNE
from sklearn.preprocessing import normalize

ROOT        = Path(__file__).parent.parent if "__file__" in dir() else Path().resolve().parent
DB_PATH     = ROOT / "database.db"
EXPORTS_DIR = ROOT / "analyses" / "exports"
EXPORTS_DIR.mkdir(parents=True, exist_ok=True)

COL_NAME        = "name"
COL_SECTOR      = "sector"
COL_COMPETITORS = "main_competitors"
RANDOM_SEED     = 42

print(f"Base: {DB_PATH}")
print(f"Exports: {EXPORTS_DIR}")

Base: C:\Users\33623\Documents\___Projets\AI\Reseaux d'acteurs\database.db
Exports: C:\Users\33623\Documents\___Projets\AI\Reseaux d'acteurs\analyses\exports


## 2. SQL Extraction of Companies and Financial Score

**Goal**: build the raw company table with competitors.

**Inputs**: `enterprises` table (`name`, `sector`, `main_competitors`, `capitalization`, `funds_raised`).
**Processing**: robust numeric parsing, then `ranking_score = COALESCE(capitalization, funds_raised)`.
**Outputs**: `df_raw`, sorted by `ranking_score` desc and filtered to positive score.

In [2]:
SQL = f"""
WITH base AS (
    SELECT
        name             AS {COL_NAME},
        sector           AS {COL_SECTOR},
        main_competitors AS {COL_COMPETITORS},

        -- Nettoie les formats numeriques heterogenes (espaces, NBSP, virgules/points).
        CASE
            WHEN NULLIF(TRIM(capitalization), '') IS NULL THEN NULL
            ELSE CASE
                WHEN (
                    LENGTH(REPLACE(REPLACE(REPLACE(TRIM(capitalization), ' ', ''), CHAR(160), ''), ',', '.'))
                    - LENGTH(REPLACE(REPLACE(REPLACE(REPLACE(TRIM(capitalization), ' ', ''), CHAR(160), ''), ',', '.'), '.', ''))
                ) > 1
                THEN CAST(REPLACE(REPLACE(REPLACE(REPLACE(TRIM(capitalization), ' ', ''), CHAR(160), ''), ',', '.'), '.', '') AS REAL)
                ELSE CAST(REPLACE(REPLACE(REPLACE(TRIM(capitalization), ' ', ''), CHAR(160), ''), ',', '.') AS REAL)
            END
        END AS cap_musd,

        CASE
            WHEN NULLIF(TRIM(funds_raised), '') IS NULL THEN NULL
            ELSE CASE
                WHEN (
                    LENGTH(REPLACE(REPLACE(REPLACE(TRIM(funds_raised), ' ', ''), CHAR(160), ''), ',', '.'))
                    - LENGTH(REPLACE(REPLACE(REPLACE(REPLACE(TRIM(funds_raised), ' ', ''), CHAR(160), ''), ',', '.'), '.', ''))
                ) > 1
                THEN CAST(REPLACE(REPLACE(REPLACE(REPLACE(TRIM(funds_raised), ' ', ''), CHAR(160), ''), ',', '.'), '.', '') AS REAL)
                ELSE CAST(REPLACE(REPLACE(REPLACE(TRIM(funds_raised), ' ', ''), CHAR(160), ''), ',', '.') AS REAL)
            END
        END AS funds_musd

    FROM enterprises
    WHERE main_competitors IS NOT NULL
      AND main_competitors != ''
)
SELECT
    {COL_NAME},
    {COL_SECTOR},
    {COL_COMPETITORS},
    COALESCE(cap_musd, funds_musd) AS ranking_score
FROM base
WHERE COALESCE(cap_musd, funds_musd) IS NOT NULL
  AND COALESCE(cap_musd, funds_musd) > 0
ORDER BY ranking_score DESC
"""

with sqlite3.connect(DB_PATH) as con:
    df_raw = pd.read_sql_query(SQL, con)

print(
    f"{len(df_raw)} entreprises retenues "
    f"(sÃ©lection: capitalisation, sinon fonds levÃ©s)"
)
df_raw[["name", "ranking_score"]].head(10)

95 entreprises retenues (sÃ©lection: capitalisation, sinon fonds levÃ©s)


,name,ranking_score
0,Nvidia,5000000.0
1,Google,4560000.0
2,Alphabet Inc.,4120000.0
3,Apple,4000000.0
4,Microsoft,3450000.0
5,Amazon,2900000.0
6,Broadcom,1835000.0
7,Meta,1503000.0
8,SpaceX,1480000.0
9,AWS,1200000.0


In [3]:
# Export brut
raw_path = EXPORTS_DIR / "competitors_raw.csv"
df_raw.to_csv(raw_path, index=False)
print(f"Export brut â†’ {raw_path}")

Export brut â†’ C:\Users\33623\Documents\___Projets\AI\Reseaux d'acteurs\analyses\exports\competitors_raw.csv


## 3. Competitor Text Cleaning

**Goal**: convert raw competitor text lists into clean structured lists.

**Processing**: multi-separator split, invalid value removal, case normalization, self-reference removal.
**Output**: `df_clean` with one competitor list per company.

In [4]:
_SEP = re.compile(r"[,;/\n]+")
_INVALID = re.compile(r"^(na|n/a|none|unknown|tbd|-)$", re.IGNORECASE)

def clean_name(s: str) -> str:
    s = s.strip().lower()
    s = re.sub(r"\(.*?\)", "", s).strip()
    s = re.sub(r"\s{2,}", " ", s)
    return s.title()

def split_competitors(raw: str) -> list[str]:
    parts = _SEP.split(str(raw))
    return [clean_name(p) for p in parts
            if clean_name(p) and not _INVALID.match(p.strip())]

df_clean = df_raw.copy()
df_clean[COL_COMPETITORS] = df_clean[COL_COMPETITORS].apply(split_competitors)

df_clean[COL_COMPETITORS] = df_clean.apply(
    lambda r: [c for c in r[COL_COMPETITORS] if c.lower() != r[COL_NAME].lower()],
    axis=1,
)

df_clean = df_clean[df_clean[COL_COMPETITORS].map(len) > 0].reset_index(drop=True)

print(f"{len(df_clean)} entreprises aprÃ¨s nettoyage")
df_clean[[COL_NAME, COL_COMPETITORS]].head(8)

95 entreprises aprÃ¨s nettoyage


,name,main_competitors
0,Nvidia,"[Amd, Intel, Google, Broadcom, Qualcomm]"
1,Google,"[Microsoft, Openai, Perpexity Ai, Amazon, Byte..."
2,Alphabet Inc.,"[Microsoft, Amazon, Apple, Meta, Oracle, Opena..."
3,Apple,"[Google, Microsoft, Samsung, Amazon, Xiaomi, H..."
4,Microsoft,"[Amazon, Google, Meta, Apple, Sony, Oracle]"
5,Amazon,"[Walmart, Temu, Shein, Alibaba, Microsoft, Goo..."
6,Broadcom,"[Nvidia, Qualcomm, Intel, Marvell Technology]"
7,Meta,"[Google, Bytedance, Snap, X, Apple, Snapchat, ..."


## 4. Semantic Name Normalization

**Goal**: merge subsidiaries, brands, and variants under canonical names.

**Inputs**: `df_clean` and `SEMANTIC_ALIASES`.
**Output**: semantically normalized `df_clean`, deduplicated, without self-competition.

In [5]:
SEMANTIC_ALIASES: dict[str, str] = {
    # â”€â”€ Google (nom canonique dans la base) â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
    "Alphabet":             "Google",
    "Alphabet Inc.":        "Google",
    "Youtube":              "Google",
    "YouTube":              "Google",
    "Deepmind":             "Google",
    "DeepMind":             "Google",
    "Google Deepmind":      "Google",
    "Google DeepMind":      "Google",
    "Google Brain":         "Google",
    "Google Cloud":         "Google",
    "Google Cloud Platform":"Google",
    "GCP":                  "Google",
    "Waymo":                "Google",
    "Verily":               "Google",
    "Calico":               "Google",
    "Waze":                 "Google",
    "Google Translate":     "Google",
    "Gmail":                "Google",
    "Android":              "Google",
    # â”€â”€ Meta (nom canonique dans la base) â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
    "Meta Platforms":       "Meta",
    "Facebook":             "Meta",
    "Instagram":            "Meta",
    "Whatsapp":             "Meta",
    "WhatsApp":             "Meta",
    "Threads":              "Meta",
    "Oculus":               "Meta",
    "Meta Quest":           "Meta",
    "LLaMA":                "Meta",
    "Llama":                "Meta",
    # â”€â”€ Microsoft â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
    "Azure":                "Microsoft",
    "Microsoft Azure":      "Microsoft",
    "Linkedin":             "Microsoft",
    "LinkedIn":             "Microsoft",
    "Github":               "Microsoft",
    "GitHub":               "Microsoft",
    "Skype":                "Microsoft",
    "Bing":                 "Microsoft",
    "Nuance":               "Microsoft",
    "Nuance Communications":"Microsoft",
    "Activision Blizzard":  "Microsoft",
    "Activision":           "Microsoft",
    "Xbox":                 "Microsoft",
    "Microsoft Translator": "Microsoft",
    "Office 365":           "Microsoft",
    # â”€â”€ Amazon â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
    "Aws":                  "AWS",
    "AWS":                  "AWS",
    "Alexa":                "Amazon",
    "Twitch":               "Amazon",
    "Amazon.Com":           "Amazon",
    "Amazon Prime":         "Amazon",
    "Amazon Prime Video":   "Amazon",
    "Kindle":               "Amazon",
    # â”€â”€ Amazon Web Services (entitÃ© sÃ©parÃ©e dans la base) â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
    "Amazon Web Services":  "AWS",
    "Amazon Web Services (AWS)": "AWS",
    # â”€â”€ Apple â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
    "Siri":                 "Apple",
    "Apple Inc.":           "Apple",
    "Apple Inc":            "Apple",
    "Iphone":               "Apple",
    "iPhone":               "Apple",
    "Ipad":                 "Apple",
    "iPad":                 "Apple",
    "Apple Silicon":        "Apple",
    # â”€â”€ Salesforce â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
    "Slack":                "Salesforce",
    "Tableau":              "Salesforce",
    "Mulesoft":             "Salesforce",
    "MuleSoft":             "Salesforce",
    "Salesforce - Einstein":"Salesforce",
    "Einstein":             "Salesforce",
    # â”€â”€ IBM â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
    "Red Hat":              "IBM",
    "RedHat":               "IBM",
    "Watsonx":              "IBM",
    "Watson":               "IBM",
    "IBM Watson":           "IBM",
    # â”€â”€ Oracle â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
    "Netsuite":             "Oracle",
    "NetSuite":             "Oracle",
    "Java":                 "Oracle",
    # â”€â”€ Nvidia (nom canonique dans la base) â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
    "NVIDIA":               "Nvidia",
    "Cuda":                 "Nvidia",
    "CUDA":                 "Nvidia",
    "Nvidia Corporation":   "Nvidia",
    # â”€â”€ INtel (nom avec cette casse dans la base) â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
    "INtel":                "Intel",
    "Intel":                "Intel",
    "Intel Corporation":    "Intel",
    # â”€â”€ AMD â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
    "Amd":                  "AMD",
    "AMD Inc.":             "AMD",
    "Advanced Micro Devices": "AMD",
    # â”€â”€ ByteDance â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
    "Tiktok":               "ByteDance",
    "TikTok":               "ByteDance",
    "Douyin":               "ByteDance",
    "Bytedance":            "ByteDance",
    "Bytedanse":            "ByteDance",
    # â”€â”€ X (nom canonique dans la base, anciennement Twitter) â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
    "Twitter":              "X",
    "X.Com":                "X",
    "X Corp":               "X",
    # â”€â”€ Tesla â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
    "Tesla Inc.":           "Tesla",
    "Tesla Motors":         "Tesla",
    # â”€â”€ SpaceX â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
    "Space Exploration Technologies": "SpaceX",
    "Starlink":             "SpaceX",
    # â”€â”€ Baidu â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
    "Ernie":                "Baidu",
    "Ernie Bot":            "Baidu",
    "ERNIE":                "Baidu",
    # â”€â”€ Tencent â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
    "Wechat":               "Tencent",
    "WeChat":               "Tencent",
    "Qq":                   "Tencent",
    "QQ":                   "Tencent",
    # â”€â”€ OpenAI â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
    "Chatgpt":              "OpenAI",
    "ChatGPT":              "OpenAI",
    "Gpt-4":                "OpenAI",
    "GPT-4":                "OpenAI",
    "Gpt4":                 "OpenAI",
    "GPT4":                 "OpenAI",
    "Openai":               "OpenAI",
    "Perpexity Ai":         "Perplexity AI",
    "DALL-E":               "OpenAI",
    "Dall-E":               "OpenAI",
    "Sora":                 "OpenAI",
    # â”€â”€ Anthropic â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
    "Claude":               "Anthropic",
    "Claude AI":            "Anthropic",
    # â”€â”€ Samsung â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
    "Samsung Electronics":  "Samsung",
    "Samsung System LSI":   "Samsung",
    # â”€â”€ Adobe â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
    "Adobe Firefly":        "Adobe",
    "Photoshop":            "Adobe",
    # â”€â”€ Qualcomm â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
    "Qualcomm Inc.":        "Qualcomm",
    # â”€â”€ SAP â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
    "SAP SE":               "SAP",
    # â”€â”€ Huawei â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
    "Huawei Technologies":  "Huawei",
    # â”€â”€ Alibaba â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
    "Alibaba Group":        "Alibaba",
    "AliCloud":             "Alibaba",
    "Alipay":               "Alibaba",
    # â”€â”€ HP â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
    "Hewlett-Packard":      "HP",
    "Hewlett Packard":      "HP",
    # â”€â”€ Netflix â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
    "Netflix Inc.":         "Netflix",
    # â”€â”€ Spotify â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
    "Spotify AB":           "Spotify",
    # â”€â”€ Sony â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
    "Sony Corporation":     "Sony",
    "PlayStation":          "Sony",
    # â”€â”€ Dell â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
    "Dell Technologies":    "Dell",
    # â”€â”€ Lenovo â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
    "Lenovo Group":         "Lenovo",
    # â”€â”€ MediaTek â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
    "MediaTek Inc.":        "MediaTek",
    # â”€â”€ Xiaomi â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
    "Xiaomi Corporation":   "Xiaomi",
    # â”€â”€ Oppo â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
    "OPPO":                 "Oppo",
    # â”€â”€ Vivo â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
    "VIVO":                 "Vivo",
    # â”€â”€ Disney â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
    "Disney+":              "Disney",
    "Walt Disney":          "Disney",
    # â”€â”€ Boeing â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
    "Boeing Company":       "Boeing",
    # â”€â”€ Autres acteurs spÃ©cifiques IA â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
    "Mistral":              "Mistral AI",
    "Stability AI":         "Stability",
    "Stability.ai":         "Stability",
    "Midjourney Inc.":      "Midjourney",
    "Runway ML":            "Runway",
}

def apply_semantic_aliases(names: list[str]) -> list[str]:
    return [SEMANTIC_ALIASES.get(n, n) for n in names]

df_clean[COL_COMPETITORS] = df_clean[COL_COMPETITORS].apply(apply_semantic_aliases)
df_clean[COL_COMPETITORS] = df_clean[COL_COMPETITORS].apply(lambda lst: list(dict.fromkeys(lst)))
df_clean[COL_COMPETITORS] = df_clean.apply(
    lambda r: [c for c in r[COL_COMPETITORS]
               if c.lower() != r[COL_NAME].lower()
               and SEMANTIC_ALIASES.get(r[COL_NAME].title(), r[COL_NAME]).lower() != c.lower()],
    axis=1,
)
df_clean = df_clean[df_clean[COL_COMPETITORS].map(len) > 0].reset_index(drop=True)

# explode garde le nom COL_COMPETITORS, pas "competitor"
preview = df_clean[[COL_NAME, COL_COMPETITORS]].explode(COL_COMPETITORS)
print(f"{len(SEMANTIC_ALIASES)} aliases | {len(df_clean)} entreprises aprÃ¨s normalisation sÃ©mantique")
print("\nTop concurrents aprÃ¨s normalisation :")
print(preview[COL_COMPETITORS].value_counts().head(15).to_string())

148 aliases | 95 entreprises aprÃ¨s normalisation sÃ©mantique

Top concurrents aprÃ¨s normalisation :
main_competitors
Google       27
Microsoft    22
OpenAI       17
Meta         13
Amazon       11
Anthropic    11
ByteDance     9
Apple         9
Nvidia        7
Intel         6
Qualcomm      6
Tencent       6
Tesla         5
Oracle        5
AMD           4


## 5. Long Directed Format `company -> competitor`

**Goal**: convert list format into row-level relational format.

**Processing**: explode, canonicalization from database names, directed pair deduplication.
**Output**: `df_long` and `competitors_long.csv`.

In [6]:
def _norm_key(value: str) -> str:
    return re.sub(r"\s+", " ", str(value).strip()).casefold()

# RÃ©fÃ©rence de canonicalisation: tous les noms d'entreprises de la base.
with sqlite3.connect(DB_PATH) as con:
    df_all_names = pd.read_sql_query("SELECT name FROM enterprises WHERE name IS NOT NULL", con)

canonical_by_key = {}
for name in df_all_names["name"].tolist():
    clean = str(name).strip()
    if clean:
        key = _norm_key(clean)
        if key not in canonical_by_key:
            canonical_by_key[key] = clean

df_long = (
    df_clean
    .explode(COL_COMPETITORS)
    .rename(columns={COL_COMPETITORS: "competitor"})
    .reset_index(drop=True)
    [[COL_NAME, "competitor", COL_SECTOR, "ranking_score"]]
)

# Canonicalise les compÃ©titeurs pour Ã©viter les doublons de casse/Ã©criture.
df_long["competitor"] = (
    df_long["competitor"]
    .map(lambda c: canonical_by_key.get(_norm_key(c), c))
)

# Force quelques formes canoniques stables, mÃªme si la base contient des variantes historiques.
preferred_canonical = {
    "INtel": "Intel",
    "Amd": "AMD",
    "Bytedanse": "ByteDance",
    "Perpexity Ai": "Perplexity AI",
    "Amazon Web Services": "AWS",
}
df_long["competitor"] = df_long["competitor"].map(lambda c: preferred_canonical.get(c, c))

# DÃ©doublonne au niveau entrepriseâ†’compÃ©titeur (dataset dirigÃ©).
df_long = (
    df_long.drop_duplicates(subset=[COL_NAME, "competitor"])
    .reset_index(drop=True)
)
df_long.insert(0, "pair_id", df_long.index)

long_path = EXPORTS_DIR / "competitors_long.csv"
df_long.to_csv(long_path, index=False)

print(f"{len(df_long)} couples entrepriseâ†’concurrent aprÃ¨s normalisation et dÃ©doublonnage")
print(f"Entreprises (lignes): {df_long[COL_NAME].nunique()} | CompÃ©titeurs (colonnes potentielles): {df_long['competitor'].nunique()}")
print(f"Export â†’ {long_path}")
df_long.head(10)

571 couples entrepriseâ†’concurrent aprÃ¨s normalisation et dÃ©doublonnage
Entreprises (lignes): 95 | CompÃ©titeurs (colonnes potentielles): 365
Export â†’ C:\Users\33623\Documents\___Projets\AI\Reseaux d'acteurs\analyses\exports\competitors_long.csv


,pair_id,name,competitor,sector,ranking_score
0,0,Nvidia,AMD,"Hardware, AI model, ICT",5000000.0
1,1,Nvidia,Intel,"Hardware, AI model, ICT",5000000.0
2,2,Nvidia,Google,"Hardware, AI model, ICT",5000000.0
3,3,Nvidia,Broadcom,"Hardware, AI model, ICT",5000000.0
4,4,Nvidia,Qualcomm,"Hardware, AI model, ICT",5000000.0
5,5,Google,Microsoft,"AI model, Media & Entertainment, Sales & Marke...",4560000.0
6,6,Google,OpenAI,"AI model, Media & Entertainment, Sales & Marke...",4560000.0
7,7,Google,Perplexity AI,"AI model, Media & Entertainment, Sales & Marke...",4560000.0
8,8,Google,Amazon,"AI model, Media & Entertainment, Sales & Marke...",4560000.0
9,9,Google,ByteDance,"AI model, Media & Entertainment, Sales & Marke...",4560000.0


## 6. Relation Aggregation

**Goal**: count frequency for each `company -> competitor` pair.

**Output**: `df_agg` (`name`, `competitor`, `count`) and `competitors_aggregated.csv`.

In [7]:
df_agg = (
    df_long
    .groupby([COL_NAME, "competitor"], sort=False)
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
    .reset_index(drop=True)
)

print("Top 20 paires entrepriseâ€“concurrent :")
display(df_agg.head(20))

print("\nDistribution des frÃ©quences :")
display(df_agg["count"].describe())

agg_path = EXPORTS_DIR / "competitors_aggregated.csv"
df_agg.to_csv(agg_path, index=False)
print(f"\nExport â†’ {agg_path}")

Top 20 paires entrepriseâ€“concurrent :


,name,competitor,count
0,Nvidia,AMD,1
1,Nvidia,Intel,1
2,Nvidia,Google,1
3,Nvidia,Broadcom,1
4,Nvidia,Qualcomm,1
5,Google,Microsoft,1
6,Google,OpenAI,1
7,Google,Perplexity AI,1
8,Google,Amazon,1
9,Google,ByteDance,1



Distribution des frÃ©quences :


count    571.0
mean       1.0
std        0.0
min        1.0
25%        1.0
50%        1.0
75%        1.0
max        1.0
Name: count, dtype: float64


Export â†’ C:\Users\33623\Documents\___Projets\AI\Reseaux d'acteurs\analyses\exports\competitors_aggregated.csv


## 7. Directed Matrix `company x competitors`

**Goal**: prepare the matrix representation used by projection and community detection.

**Rows**: selected companies.
**Columns**: mentioned competitors.
**Output**: `df_matrix` and `cooccurrence_matrix.csv` (directed matrix).

In [8]:
# Matrice dirigÃ©e: lignes = entreprises, colonnes = compÃ©titeurs.
# On garde la logique asymÃ©trique, sans M + M^T.
df_matrix = df_agg.pivot_table(
    index=COL_NAME,
    columns="competitor",
    values="count",
    fill_value=0,
)

enterprise_actors = sorted(df_raw[COL_NAME].dropna().astype(str).str.strip().unique())
df_matrix = df_matrix.reindex(index=enterprise_actors, fill_value=0)

print(f"Matrice dirigÃ©e: {df_matrix.shape[0]} entreprises Ã— {df_matrix.shape[1]} compÃ©titeurs")
print(f"DensitÃ© non-nulle : {(df_matrix.values > 0).mean():.1%}")

cooc_path = EXPORTS_DIR / "cooccurrence_matrix.csv"
df_matrix.to_csv(cooc_path)
print(f"Export matrice dirigÃ©e â†’ {cooc_path}")

Matrice dirigÃ©e: 95 entreprises Ã— 365 compÃ©titeurs
DensitÃ© non-nulle : 1.6%
Export matrice dirigÃ©e â†’ C:\Users\33623\Documents\___Projets\AI\Reseaux d'acteurs\analyses\exports\cooccurrence_matrix.csv


## 8. 3D Projection (t-SNE)

**Goal**: project company competition profiles into a richer latent geometry.

**Input**: L2-normalized `df_matrix`.
**Key parameter**: `perplexity = 20`.
**Output**: `df_coords` with (`x`, `y`, `z`) and export `coords_3d.csv`.

In [9]:
N = df_matrix.shape[0]
X = normalize(df_matrix.values, norm="l2")

perplexity = 20
tsne_dims = 3
print(f"t-SNE {tsne_dims}D on {N} companies (perplexity={perplexity})")

coords = TSNE(
    n_components=tsne_dims,
    perplexity=perplexity,
    init="pca",
    learning_rate="auto",
    metric="cosine",
    random_state=RANDOM_SEED,
).fit_transform(X)

method = "t-SNE-3D"

out_degree = df_matrix.sum(axis=1).to_dict()
ranking_score_map = df_raw.set_index(COL_NAME)["ranking_score"].to_dict()

enterprise_names = list(df_matrix.index)
df_coords = pd.DataFrame({
    "actor": enterprise_names,
    "x": coords[:, 0],
    "y": coords[:, 1],
    "z": coords[:, 2],
})

df_coords["score"] = df_coords["actor"].map(out_degree).fillna(0)
df_coords["log_score"] = np.log10(df_coords["score"] + 1)
df_coords["ranking_score"] = df_coords["actor"].map(ranking_score_map).fillna(0)

sector_map = (
    df_raw.set_index(COL_NAME)[COL_SECTOR]
    .dropna()
    .apply(lambda s: s.split(",")[0].strip())
    .to_dict()
)
df_coords["sector"] = df_coords["actor"].map(sector_map).fillna("Unknown")

coords_path = EXPORTS_DIR / "coords_3d.csv"
df_coords.to_csv(coords_path, index=False)
print(f"3D coordinates ({method}) -> {coords_path}")
df_coords.sort_values("ranking_score", ascending=False).head(10)

t-SNE 3D on 95 companies (perplexity=20)


c:\Users\33623\Documents\___Projets\AI\Reseaux d'acteurs\.venv\Lib\site-packages\joblib\externals\loky\backend\context.py:131: UserWarning: Could not find the number of physical cores for the following reason:
[WinError 2] Le fichier spécifié est introuvable
Returning the number of logical cores instead. You can silence this warning by setting LOKY_MAX_CPU_COUNT to the number of cores you want to use.
  warnings.warn(
  File "c:\Users\33623\Documents\___Projets\AI\Reseaux d'acteurs\.venv\Lib\site-packages\joblib\externals\loky\backend\context.py", line 247, in _count_physical_cores
    cpu_count_physical = _count_physical_cores_win32()
                         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\33623\Documents\___Projets\AI\Reseaux d'acteurs\.venv\Lib\site-packages\joblib\externals\loky\backend\context.py", line 299, in _count_physical_cores_win32
    cpu_info = subprocess.run(
               ^^^^^^^^^^^^^^^
  File "C:\Users\33623\AppData\Local\Programs\Python\Python312\Lib

3D coordinates (t-SNE-3D) -> C:\Users\33623\Documents\___Projets\AI\Reseaux d'acteurs\analyses\exports\coords_3d.csv


,actor,x,y,z,score,log_score,ranking_score,sector
62,Nvidia,-3.512296,-43.207527,50.201241,5.0,0.778151,5000000.0,Hardware
47,Google,64.065269,-16.284067,-81.738045,8.0,0.954243,4560000.0,AI model
12,Alphabet Inc.,42.445431,-3.718408,-81.157654,10.0,1.041393,4120000.0,Cloud Provider
19,Apple,-10.445779,-87.029175,-18.384212,16.0,1.230449,4000000.0,ICT
58,Microsoft,-5.399166,-29.581255,-30.988256,6.0,0.845098,3450000.0,Cloud Provider
14,Amazon,17.730989,-90.304001,-32.073452,10.0,1.041393,2900000.0,Unknown
25,Broadcom,-34.302094,-53.590893,41.276104,4.0,0.698970,1835000.0,Hardware
57,Meta,-8.987450,-16.207458,-56.955368,12.0,1.113943,1503000.0,AI model
83,SpaceX,-0.519145,9.184302,-89.035873,8.0,0.954243,1480000.0,Cloud Provider
7,AWS,18.538471,-6.039607,56.291214,5.0,0.778151,1200000.0,Hardware


## 9. Final Community Map (Louvain on 3D Coordinates, 2D Rendering)

**Goal**: produce the final map with graph-based communities.

**Method**:
- Build a k-nearest-neighbors graph from 3D coordinates (`x`, `y`, `z`).
- Weight edges by distance similarity.
- Run Louvain community detection on this graph.
- Refine tiny communities by local nearest-neighbor reassignment.
- Render labels and blobs in 2D (`x`, `y`) for readability.

**Visual rules**:
- Label color by country (custom country palette).
- Label size from `ranking_score` in `sqrt` mode (range 8-18).
- Blob color follows mean `z` per community (yellow -> green gradient).
- Labels only (no point markers).

**Outputs**:
- `communities_kmeans_2d.csv` (community assignments)
- `community_labels_short.csv` (2-3 word community labels)
- `competition_map_2d_kmeans.html` (interactive 2D final map)

In [10]:
import plotly.graph_objects as go

info_cols = [
    "founded_year", "employees_count", "revenue_millions",
    "capitalization", "funds_raised", "description"
]

# Strategie de taille des labels (solution recommandee)
SIZE_MODE = "sqrt"  # options: linear, linear_clip_p99, log10, log10_clip_p99, log1p, log1p_clip_p99, sqrt, sqrt_clip_p99
SIZE_MIN, SIZE_MAX = 8, 18
SIZE_FALLBACK = 10
SIZE_CLIP_Q = 0.99


def country_key(value: str) -> str:
    return str(value).strip().casefold()


def scale_to_range(values: pd.Series, min_size: float, max_size: float, fallback_size: float) -> pd.Series:
    if values.isna().all() or np.isclose(values.max(), values.min()):
        return pd.Series(fallback_size, index=values.index)
    scaled = min_size + (values - values.min()) * (max_size - min_size) / (values.max() - values.min())
    return scaled


def compute_label_sizes(
    raw_scores: pd.Series,
    mode: str,
    min_size: float,
    max_size: float,
    fallback_size: float,
    clip_q: float,
    ) -> tuple[pd.Series, dict]:
    scores = pd.to_numeric(raw_scores, errors="coerce").where(lambda s: s > 0)
    if not scores.notna().any():
        return pd.Series(fallback_size, index=raw_scores.index), {"mode": mode, "clip_value": None}

    use_clip = "_clip_p99" in mode
    base_mode = mode.replace("_clip_p99", "")

    working = scores.copy()
    clip_value = None
    if use_clip:
        clip_value = float(working.quantile(clip_q))
        working = working.clip(upper=clip_value)

    if base_mode == "linear":
        transformed = working
    elif base_mode == "log10":
        transformed = np.log10(working)
    elif base_mode == "log1p":
        transformed = np.log1p(working)
    elif base_mode == "sqrt":
        transformed = np.sqrt(working)
    else:
        raise ValueError(f"SIZE_MODE inconnu: {mode}")

    sizes = scale_to_range(transformed, min_size=min_size, max_size=max_size, fallback_size=fallback_size).round(1)
    return sizes, {"mode": mode, "clip_value": clip_value}


def apply_label_repel(
    df: pd.DataFrame,
    x_col: str = "x",
    y_col: str = "y",
    size_col: str = "label_size",
    iterations: int = 320,
    anchor_pull: float = 0.02,
    step: float = 0.55,
    ) -> pd.DataFrame:
    """
    Petit moteur de repulsion (type force layout) pour limiter les chevauchements de labels.
    La simulation tourne en coordonnees normalisees [0,1] puis revient en coordonnees d'origine.
    """
    x = pd.to_numeric(df[x_col], errors="coerce").to_numpy(dtype=float)
    y = pd.to_numeric(df[y_col], errors="coerce").to_numpy(dtype=float)
    s = pd.to_numeric(df[size_col], errors="coerce").fillna(SIZE_FALLBACK).to_numpy(dtype=float)

    n = len(df)
    if n <= 1:
        return pd.DataFrame({"x_label": x, "y_label": y}, index=df.index)

    xmin, xmax = float(np.min(x)), float(np.max(x))
    ymin, ymax = float(np.min(y)), float(np.max(y))
    xspan = xmax - xmin if not np.isclose(xmax, xmin) else 1.0
    yspan = ymax - ymin if not np.isclose(ymax, ymin) else 1.0

    pos = np.column_stack([(x - xmin) / xspan, (y - ymin) / yspan])
    origin = pos.copy()

    smin, smax = float(np.min(s)), float(np.max(s))
    sspan = smax - smin if not np.isclose(smax, smin) else 1.0
    radii = 0.010 + ((s - smin) / sspan) * 0.028

    for _ in range(iterations):
        disp = np.zeros_like(pos)

        for i in range(n - 1):
            delta = pos[i] - pos[i + 1 :]
            dist = np.linalg.norm(delta, axis=1) + 1e-9
            target = radii[i] + radii[i + 1 :]
            overlap = target - dist
            mask = overlap > 0

            if np.any(mask):
                force = (overlap[mask] / dist[mask])[:, None] * delta[mask]
                move = force * step
                disp[i] += move.sum(axis=0)
                disp[i + 1 :][mask] -= move

        disp += (origin - pos) * anchor_pull
        pos += disp
        pos = np.clip(pos, 0.0, 1.0)

    x_label = pos[:, 0] * xspan + xmin
    y_label = pos[:, 1] * yspan + ymin
    return pd.DataFrame({"x_label": x_label, "y_label": y_label}, index=df.index)


def build_country_colors(countries: list[str]) -> dict[str, str]:
    forced_colors = {
        "china": "#D62828",                     # rouge
        "united states": "#1D4ED8",             # bleu
        "united states of america": "#1D4ED8",  # bleu
        "usa": "#1D4ED8",                       # bleu
        "canada": "#60A5FA",                    # bleu clair
        "japan": "#EC4899",                     # rose
        "south korea": "#EC4899",               # rose
        "korea, south": "#EC4899",              # rose
        "republic of korea": "#EC4899",         # rose
    }

    europe_country_keys = {
        "albania", "andorra", "austria", "belarus", "belgium", "bosnia and herzegovina",
        "bulgaria", "croatia", "cyprus", "czechia", "czech republic", "denmark", "estonia",
        "finland", "france", "germany", "greece", "hungary", "iceland", "ireland", "italy",
        "latvia", "liechtenstein", "lithuania", "luxembourg", "malta", "moldova", "monaco",
        "montenegro", "netherlands", "north macedonia", "norway", "poland", "portugal", "romania",
        "san marino", "serbia", "slovakia", "slovenia", "spain", "sweden", "switzerland",
        "ukraine", "united kingdom", "vatican city", "kosovo"
    }

    greens = [
        "#1B4332", "#24553F", "#2D6A4F", "#3A7D5D", "#4C956C", "#5FAF7D",
        "#74C69D", "#52B788", "#40916C", "#3E8E63", "#2F7D57", "#2A6F54"
    ]

    fallback_palette = [
        "#6D597A", "#E76F51", "#264653", "#457B9D", "#BC6C25", "#B56576",
        "#3A86FF", "#FF006E", "#0A9396", "#7F5539", "#4361EE", "#FF7F11",
        "#2B2D42", "#8D99AE", "#8338EC", "#3D405B"
    ]

    color_map = {}
    europe = sorted([c for c in countries if country_key(c) in europe_country_keys])
    non_europe = [c for c in countries if c not in europe]

    for i, country in enumerate(europe):
        color_map[country] = greens[i % len(greens)]

    fallback_idx = 0
    for country in non_europe:
        key = country_key(country)
        if key in forced_colors:
            color_map[country] = forced_colors[key]
        else:
            color_map[country] = fallback_palette[fallback_idx % len(fallback_palette)]
            fallback_idx += 1

    return color_map


def format_hover(row: pd.Series) -> str:
    lines = [f"<b>{row['actor']}</b>"]

    if pd.notna(row.get("sector")):
        lines.append(f"Sector: {row['sector']}")
    if pd.notna(row.get("country")):
        lines.append(f"Country: {row['country']}")
    if pd.notna(row.get("founded_year")):
        lines.append(f"Founded: {int(row['founded_year'])}")
    if pd.notna(row.get("employees_count")):
        lines.append(f"Employees: {int(row['employees_count']):,}")

    cap = row.get("capitalization_num")
    if pd.notna(cap) and cap > 0:
        lines.append(f"Market cap: {cap/1000:.1f}B USD")

    rev = row.get("revenue_num")
    if pd.notna(rev) and rev > 0:
        lines.append(f"Revenue: {rev/1000:.1f}B USD")

    if pd.notna(row.get("description")):
        desc = str(row["description"])
        snippet = desc[:160].rstrip()
        lines.append(f"<i>{snippet}{'â€¦' if len(desc) > 160 else ''}</i>")

    lines.append(f"Outgoing competitor links: {int(row['score'])}")
    return "<br>".join(lines)


# Assemble les donnees de plotting.
df_plot = df_coords.copy()
available = [c for c in info_cols if c in df_raw.columns]
if available:
    df_info = df_raw.set_index(COL_NAME)[available]
    for col in available:
        df_plot[col] = df_plot["actor"].map(df_info[col])

# Ajoute le pays avec fallback SQL si la colonne manque dans df_raw.
if "country" in df_raw.columns:
    country_map = df_raw.set_index(COL_NAME)["country"]
else:
    with sqlite3.connect(DB_PATH) as con:
        df_country = pd.read_sql_query(f"SELECT name AS {COL_NAME}, country FROM enterprises", con)
    country_map = df_country.set_index(COL_NAME)["country"]

df_plot["country"] = df_plot["actor"].map(country_map).fillna("Unknown")

# Colonnes numeriques utiles pour hover/tailles.
df_plot["capitalization_num"] = pd.to_numeric(df_plot["capitalization"], errors="coerce") if "capitalization" in df_plot.columns else np.nan
df_plot["revenue_num"] = pd.to_numeric(df_plot["revenue_millions"], errors="coerce") if "revenue_millions" in df_plot.columns else np.nan

# Taille des labels: mode configurable, defaut log1p + clipping P99.
df_plot["label_size"], size_meta = compute_label_sizes(
    raw_scores=df_plot["ranking_score"],
    mode=SIZE_MODE,
    min_size=SIZE_MIN,
    max_size=SIZE_MAX,
    fallback_size=SIZE_FALLBACK,
    clip_q=SIZE_CLIP_Q,
    )

# Repositionnement repel en deux passes: global puis micro-ajustement intra-pays.
label_xy = apply_label_repel(
    df_plot,
    x_col="x",
    y_col="y",
    size_col="label_size",
    iterations=380,
    anchor_pull=0.018,
    step=0.62,
    )
df_plot["x_label"] = label_xy["x_label"]
df_plot["y_label"] = label_xy["y_label"]

for country_name, idx in df_plot.groupby("country").groups.items():
    sub = df_plot.loc[idx, ["x_label", "y_label", "label_size"]].rename(
        columns={"x_label": "x", "y_label": "y"}
    )
    sub_xy = apply_label_repel(
        sub,
        x_col="x",
        y_col="y",
        size_col="label_size",
        iterations=160,
        anchor_pull=0.045,
        step=0.36,
    )
    df_plot.loc[idx, "x_label"] = sub_xy["x_label"].to_numpy()
    df_plot.loc[idx, "y_label"] = sub_xy["y_label"].to_numpy()

mean_shift = np.mean(np.sqrt((df_plot["x_label"] - df_plot["x"]) ** 2 + (df_plot["y_label"] - df_plot["y"]) ** 2))
print(f"Repel shift moyen: {mean_shift:.2f} (unitÃ©s de projection)")
print(f"Label size stats -> min: {df_plot['label_size'].min():.1f}, max: {df_plot['label_size'].max():.1f}")
print(f"Label size basis -> {size_meta['mode']} on ranking_score")
if size_meta["clip_value"] is not None:
    print(f"Clipping upper bound (P99): {size_meta['clip_value']:.3g}")

# Hover + couleurs par pays.
df_plot["hover"] = df_plot.apply(format_hover, axis=1)
countries = sorted(df_plot["country"].dropna().unique().tolist())
country_colors = build_country_colors(countries)

fig = go.Figure()
for country in countries:
    sub = df_plot[df_plot["country"] == country].copy().sort_values("label_size", ascending=True)

    sub["label_text"] = sub["actor"].combine(sub["label_size"], lambda actor, size: (
        f"<span style='font-size:{size}px;font-weight:400;"
        "text-shadow:0 0 1px rgba(253,250,244,0.95),0 0 3px rgba(253,250,244,0.65)'>"
        f"{actor}</span>"
    ))

    fig.add_trace(
        go.Scatter(
            x=sub["x_label"],
            y=sub["y_label"],
            mode="text",
            text=sub["label_text"],
            textposition="middle center",
            textfont=dict(color=country_colors[country]),
            name=country,
            customdata=np.stack([sub["hover"]], axis=-1),
            hovertemplate="%{customdata[0]}<extra></extra>",
            showlegend=True,
        )
    )

fig.update_layout(
    title=f"{method} â€” Entreprises dans l'espace des competiteurs ({N} entreprises)",
    xaxis=dict(title="Dimension 1", showgrid=False, zeroline=False),
    yaxis=dict(title="Dimension 2", showgrid=False, zeroline=False),
    legend=dict(title="Pays", font=dict(size=11)),
    font=dict(family="Inter, sans-serif", size=12),
    plot_bgcolor="#FDFAF4",
    paper_bgcolor="#FDFAF4",
    hovermode="closest",
    width=1400,
    height=1200,
    )

fig_html = EXPORTS_DIR / "competition_map_2d.html"
fig.write_html(str(fig_html))
print(f"Carte interactive ({method}) -> {fig_html}")
fig.show()

Repel shift moyen: 0.75 (unitÃ©s de projection)
Label size stats -> min: 8.0, max: 18.0
Label size basis -> sqrt on ranking_score
Carte interactive (t-SNE-3D) -> C:\Users\33623\Documents\___Projets\AI\Reseaux d'acteurs\analyses\exports\competition_map_2d.html


In [24]:
import networkx as nx
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler

# Community detection uses t-SNE 3D geometry, while map rendering stays on (x, y).
n_points = len(df_plot)
coords_for_graph = ["x", "y", "z"] if "z" in df_plot.columns else ["x", "y"]
XYZ = df_plot[coords_for_graph].to_numpy(dtype=float)

# Louvain settings on a weighted k-NN graph (very fine segmentation).
k_neighbors = 6
louvain_resolution = 1.75
min_refined_size = 1

if n_points <= 1:
    cluster_idx = np.zeros(n_points, dtype=int)
else:
    XYZ_scaled = StandardScaler().fit_transform(XYZ)
    k_eff = min(max(2, k_neighbors), max(1, n_points - 1))

    knn = NearestNeighbors(n_neighbors=k_eff + 1, metric="euclidean")
    knn.fit(XYZ_scaled)
    distances, indices = knn.kneighbors(XYZ_scaled)

    sigma = float(np.median(distances[:, 1:])) if distances.shape[1] > 1 else 1.0
    if not np.isfinite(sigma) or sigma <= 0:
        sigma = 1.0

    G = nx.Graph()
    G.add_nodes_from(range(n_points))

    for i in range(n_points):
        for j, d in zip(indices[i, 1:], distances[i, 1:]):
            j = int(j)
            if i == j:
                continue

            w = float(np.exp(-(d ** 2) / (2 * sigma ** 2)))
            if G.has_edge(i, j):
                if w > G[i][j]["weight"]:
                    G[i][j]["weight"] = w
            else:
                G.add_edge(i, j, weight=w)

    if G.number_of_edges() == 0:
        cluster_idx = np.arange(n_points, dtype=int)
    else:
        communities = nx.community.louvain_communities(
            G,
            weight="weight",
            resolution=louvain_resolution,
            seed=RANDOM_SEED,
        )

        cluster_idx = np.full(n_points, -1, dtype=int)
        for c_id, nodes in enumerate(communities):
            for node in nodes:
                cluster_idx[int(node)] = int(c_id)

        # Refinement: reassign only singleton communities to nearest stronger communities.
        sizes = pd.Series(cluster_idx).value_counts().to_dict()
        tiny_ids = {int(c) for c, sz in sizes.items() if int(sz) < min_refined_size}
        if tiny_ids:
            for node in np.where(np.isin(cluster_idx, list(tiny_ids)))[0]:
                for neigh in indices[int(node), 1:]:
                    neigh = int(neigh)
                    c_neigh = int(cluster_idx[neigh])
                    if c_neigh not in tiny_ids:
                        cluster_idx[int(node)] = c_neigh
                        break

        # Safety fallback in case a node remains unlabeled.
        unlabeled = np.where(cluster_idx < 0)[0]
        if len(unlabeled):
            next_id = int(cluster_idx.max() + 1) if np.any(cluster_idx >= 0) else 0
            for node in unlabeled:
                cluster_idx[int(node)] = next_id
                next_id += 1

        # Stable remap to 0..K-1 after refinement.
        uniq = sorted(np.unique(cluster_idx).tolist())
        remap = {old: new for new, old in enumerate(uniq)}
        cluster_idx = np.array([remap[int(c)] for c in cluster_idx], dtype=int)

df_plot["community"] = cluster_idx.astype(int)
community_method = (
    f"louvain_3d_to_2d(k_neighbors={k_neighbors},"
    f"resolution={louvain_resolution},min_refined_size={min_refined_size})"
)
community_values = sorted(c for c in df_plot["community"].unique() if c >= 0)


def infer_cluster_label(sub: pd.DataFrame) -> str:
    sector_text = " ".join(sub["sector"].fillna("Unknown").astype(str).str.lower().tolist())
    actor_text = " ".join(sub["actor"].fillna("").astype(str).str.lower().tolist())

    taxonomy = [
        ("AI Chip Infrastructure", ["hardware", "chip", "semiconductor", "gpu", "processor"]),
        ("AI Cloud Platforms", ["cloud", "infrastructure", "platform", "provider", "saas"]),
        ("Foundation Model Ecosystem", ["ai model", "foundation", "llm", "generative", "language model"]),
        ("AI Health and Biotech", ["health", "medical", "biotech", "pharma", "drug"]),
        ("AI Finance and Data", ["finance", "fintech", "bank", "payment", "insur"]),
        ("AI Mobility and Robotics", ["robot", "autonomous", "mobility", "vehicle", "drone"]),
    ]

    best_label = "Applied AI Mix"
    best_score = -1
    for label, keywords in taxonomy:
        score = sum(sector_text.count(k) + actor_text.count(k) for k in keywords)
        if score > best_score:
            best_label = label
            best_score = score

    return best_label


def anchor_actor(sample_actors: str) -> str:
    actor = str(sample_actors).split(",")[0].strip()
    if not actor:
        return "Core"
    return actor


def country_code(country: str) -> str:
    c = str(country).strip().lower()
    mapping = {
        "united states": "US",
        "united states of america": "US",
        "usa": "US",
        "united kingdom": "UK",
        "germany": "DE",
        "france": "FR",
        "china": "CN",
        "japan": "JP",
        "south korea": "KR",
        "unknown": "UNK",
    }
    return mapping.get(c, c[:3].upper() if c else "UNK")


def disambiguate_labels(df_labels: pd.DataFrame) -> pd.DataFrame:
    out = df_labels.copy()
    counts = out["label_short"].value_counts()
    dup_labels = counts[counts > 1].index.tolist()

    for label in dup_labels:
        idxs = out.index[out["label_short"] == label].tolist()
        for i in idxs:
            actor = anchor_actor(out.loc[i, "sample_actors"])
            out.loc[i, "label_short"] = f"{label} - {actor}"

    # If duplicates remain, append country code.
    counts2 = out["label_short"].value_counts()
    dup2 = counts2[counts2 > 1].index.tolist()
    for label in dup2:
        idxs = out.index[out["label_short"] == label].tolist()
        for i in idxs:
            ctag = country_code(out.loc[i, "top_country"])
            out.loc[i, "label_short"] = f"{label} ({ctag})"

    # Final uniqueness safeguard.
    seen = {}
    for i, label in out["label_short"].items():
        seen[label] = seen.get(label, 0) + 1
        if seen[label] > 1:
            out.loc[i, "label_short"] = f"{label} {seen[label]}"

    return out


# Short interpretation table saved as CSV.
rows = []
group_name_map = {}
for c in community_values:
    sub_c = df_plot[df_plot["community"] == c].copy()
    label_short = infer_cluster_label(sub_c)

    top_country = sub_c["country"].fillna("Unknown").value_counts().index[0] if len(sub_c) else "Unknown"
    top_sector = sub_c["sector"].fillna("Unknown").value_counts().index[0] if len(sub_c) else "Unknown"
    sample_actors = ", ".join(sub_c.sort_values("ranking_score", ascending=False)["actor"].head(5).tolist())

    rows.append({
        "community": int(c),
        "label_short": label_short,
        "n_actors": int(len(sub_c)),
        "top_country": top_country,
        "top_sector": top_sector,
        "sample_actors": sample_actors,
    })

df_cluster_labels = pd.DataFrame(rows).sort_values("community").reset_index(drop=True)
df_cluster_labels = disambiguate_labels(df_cluster_labels)
for _, r in df_cluster_labels.iterrows():
    group_name_map[int(r["community"])] = str(r["label_short"])

interpretation_path = EXPORTS_DIR / "community_labels_short.csv"
df_cluster_labels.to_csv(interpretation_path, index=False)

communities_path = EXPORTS_DIR / "communities_kmeans_2d.csv"
df_plot[["actor", "community", "country", "sector", "ranking_score", "x", "y", "z"]].sort_values(
    ["community", "ranking_score"], ascending=[True, False]
).to_csv(communities_path, index=False)

community_sizes = df_plot.groupby("community").size().sort_values(ascending=False)
main_communities = [int(c) for c, size in community_sizes.items() if c >= 0 and int(size) >= 2]


def hex_to_rgba(hex_color: str, alpha: float) -> str:
    h = hex_color.lstrip("#")
    r = int(h[0:2], 16)
    g = int(h[2:4], 16)
    b = int(h[4:6], 16)
    return f"rgba({r},{g},{b},{alpha})"


def lerp_color(hex_a: str, hex_b: str, t: float) -> str:
    t = float(np.clip(t, 0.0, 1.0))
    a = hex_a.lstrip("#")
    b = hex_b.lstrip("#")
    ar, ag, ab = int(a[0:2], 16), int(a[2:4], 16), int(a[4:6], 16)
    br, bg, bb = int(b[0:2], 16), int(b[2:4], 16), int(b[4:6], 16)
    rr = int(round(ar + (br - ar) * t))
    rg = int(round(ag + (bg - ag) * t))
    rb = int(round(ab + (bb - ab) * t))
    return f"#{rr:02X}{rg:02X}{rb:02X}"


z_mean_by_community = df_plot.groupby("community")["z"].mean().to_dict() if "z" in df_plot.columns else {}
if community_values and z_mean_by_community:
    z_vals = np.array([z_mean_by_community[c] for c in community_values], dtype=float)
    z_min, z_max = float(np.min(z_vals)), float(np.max(z_vals))
else:
    z_min, z_max = 0.0, 1.0

if np.isclose(z_max, z_min):
    community_colors = {c: "#84CC16" for c in community_values}
else:
    community_colors = {}
    for c in community_values:
        zc = float(z_mean_by_community.get(c, z_min))
        t = (zc - z_min) / (z_max - z_min)
        community_colors[c] = lerp_color("#FDE047", "#166534", t)


def convex_hull(points: np.ndarray) -> np.ndarray:
    pts = np.unique(points, axis=0)
    if len(pts) <= 2:
        return pts

    pts = pts[np.lexsort((pts[:, 1], pts[:, 0]))]

    def cross(o, a, b):
        return (a[0] - o[0]) * (b[1] - o[1]) - (a[1] - o[1]) * (b[0] - o[0])

    lower = []
    for p in pts:
        while len(lower) >= 2 and cross(lower[-2], lower[-1], p) <= 0:
            lower.pop()
        lower.append(tuple(p))

    upper = []
    for p in pts[::-1]:
        while len(upper) >= 2 and cross(upper[-2], upper[-1], p) <= 0:
            upper.pop()
        upper.append(tuple(p))

    return np.array(lower[:-1] + upper[:-1], dtype=float)


def chaikin_smooth(closed_poly: np.ndarray, refinements: int = 2) -> np.ndarray:
    poly = closed_poly.copy()
    for _ in range(refinements):
        new_pts = []
        for i in range(len(poly) - 1):
            p0 = poly[i]
            p1 = poly[i + 1]
            q = 0.75 * p0 + 0.25 * p1
            r = 0.25 * p0 + 0.75 * p1
            new_pts.extend([q, r])
        poly = np.vstack([new_pts, new_pts[0]])
    return poly


def build_group_patch(sub: pd.DataFrame, pad: float) -> tuple[np.ndarray, np.ndarray, float, float]:
    pts = sub[["x_label", "y_label"]].to_numpy(dtype=float)
    cx = float(np.mean(pts[:, 0]))
    cy = float(np.mean(pts[:, 1]))

    if len(pts) >= 3:
        hull = convex_hull(pts)
        if len(hull) >= 3:
            vec = hull - np.array([cx, cy])
            dist_local = np.linalg.norm(vec, axis=1, keepdims=True)
            scale = (dist_local + pad) / np.maximum(dist_local, 1e-6)
            inflated = np.array([cx, cy]) + vec * scale
            closed = np.vstack([inflated, inflated[0]])
            smooth = chaikin_smooth(closed, refinements=2)
            return smooth[:, 0], smooth[:, 1], cx, cy

    r = max(0.7, pad)
    theta = np.linspace(0, 2 * np.pi, 90)
    x_blob = cx + (r * 1.15) * np.cos(theta)
    y_blob = cy + (r * 0.90) * np.sin(theta)
    return x_blob, y_blob, cx, cy


fig_louvain = go.Figure()
community_centroids = {}

for c in main_communities:
    sub = df_plot[df_plot["community"] == c]
    if len(sub) < 2:
        continue

    pad = float(max(0.8, sub["label_size"].mean() * 0.24))
    x_blob, y_blob, cx, cy = build_group_patch(sub, pad=pad)

    community_centroids[c] = (cx, cy, int(len(sub)))
    color = community_colors.get(c, "#84CC16")
    label_name = group_name_map.get(int(c), f"G{c}")

    fig_louvain.add_trace(
        go.Scatter(
            x=x_blob,
            y=y_blob,
            mode="lines",
            fill="toself",
            fillcolor=hex_to_rgba(color, 0.18),
            line=dict(color=hex_to_rgba(color, 0.78), width=2.0),
            name=f"Zone {label_name}",
            hovertemplate=f"{label_name}<br>Group {c}<br>Actors: {len(sub)}<extra></extra>",
            showlegend=False,
        )
    )

if "country_colors" not in globals():
    countries = sorted(df_plot["country"].dropna().unique().tolist())
    country_colors = build_country_colors(countries)

countries = sorted(df_plot["country"].dropna().unique().tolist())
for country in countries:
    sub = df_plot[df_plot["country"] == country].copy().sort_values("label_size", ascending=True)

    label_sizes = (sub["label_size"].astype(float) * 1.55).clip(lower=12, upper=30).round(1)

    fig_louvain.add_trace(
        go.Scatter(
            x=sub["x_label"],
            y=sub["y_label"],
            mode="text",
            text=sub["actor"],
            textposition="middle center",
            textfont=dict(
                color=country_colors.get(country, "#374151"),
                size=label_sizes.to_list(),
                family="Inter, Arial, sans-serif",
            ),
            name=country,
            customdata=np.stack([sub["hover"], sub["community"]], axis=-1),
            hovertemplate="%{customdata[0]}<br>Group: %{customdata[1]}<extra></extra>",
            showlegend=True,
        )
    )

community_annotations = []
for c, (cx, cy, n_members) in community_centroids.items():
    label_name = group_name_map.get(int(c), f"G{c}")
    community_annotations.append(
        dict(
            x=cx,
            y=cy,
            text=f"{label_name}<br>(n={n_members})",
            showarrow=False,
            font=dict(size=16, color="#111827"),
            bgcolor="rgba(253,250,244,0.90)",
            bordercolor=hex_to_rgba(community_colors.get(c, "#84CC16"), 0.82),
            borderwidth=1,
            borderpad=5,
        )
    )

fig_louvain.update_layout(
    title=f"{method} - Communities on 2D map ({community_method})",
    xaxis=dict(title=dict(text="Dimension 1", font=dict(size=18)), tickfont=dict(size=13), showgrid=False, zeroline=False),
    yaxis=dict(title=dict(text="Dimension 2", font=dict(size=18)), tickfont=dict(size=13), showgrid=False, zeroline=False),
    legend=dict(title="Country", title_font=dict(size=15), font=dict(size=13)),
    font=dict(family="Inter, Arial, sans-serif", size=14),
    title_font=dict(size=26),
    plot_bgcolor="#FDFAF4",
    paper_bgcolor="#FDFAF4",
    hovermode="closest",
    width=1600,
    height=1300,
    annotations=community_annotations,
)

fig_louvain_html = EXPORTS_DIR / "competition_map_2d_kmeans.html"
fig_louvain.write_html(str(fig_louvain_html))

png_path = EXPORTS_DIR / "competition_map_2d.png"
try:
    fig_louvain.write_image(str(png_path), width=2200, height=1700, scale=2)
    print(f"PNG export -> {png_path}")
except Exception as e:
    print(f"PNG export skipped ({type(e).__name__}): {e}")

n_noise = int((df_plot["community"] < 0).sum())
print(f"Groups detected: {len(community_values)} ({community_method})")
print(f"Noise points: {n_noise}")
print(f"Blob groups (size >= 2): {len(main_communities)}")
print(f"Interpretation CSV -> {interpretation_path}")
print(f"Group export -> {communities_path}")
print(f"Map export -> {fig_louvain_html}")
display(df_cluster_labels)
fig_louvain.show()

PNG export skipped (ValueError): 
Image export using the "kaleido" engine requires the Kaleido package,
which can be installed using pip:

    $ pip install --upgrade kaleido

Groups detected: 9 (louvain_3d_to_2d(k_neighbors=6,resolution=1.75,min_refined_size=1))
Noise points: 0
Blob groups (size >= 2): 9
Interpretation CSV -> C:\Users\33623\Documents\___Projets\AI\Reseaux d'acteurs\analyses\exports\community_labels_short.csv
Group export -> C:\Users\33623\Documents\___Projets\AI\Reseaux d'acteurs\analyses\exports\communities_kmeans_2d.csv
Map export -> C:\Users\33623\Documents\___Projets\AI\Reseaux d'acteurs\analyses\exports\competition_map_2d_kmeans.html


,community,label_short,n_actors,top_country,top_sector,sample_actors
0,0,AI Cloud Platforms,12,United States,AI model,"Meta, SpaceX, YouTube, Alibaba, OpenAI"
1,1,Foundation Model Ecosystem - Apple,8,United States,AI model,"Apple, Microsoft, Tesla, Wayve, Ineffable Inte..."
2,2,AI Health and Biotech - Abridge,7,United States,Health & Social Care,"Abridge, Doctolib, Job&Talent, Ambiance Health..."
3,3,Foundation Model Ecosystem - Adobe,9,United States,AI model,"Adobe, Cloudflare, Anysphere, GitHub, Cognitio..."
4,4,AI Mobility and Robotics,12,United States,Robotics,"Samsung Electronics, Intel, Oracle, Figure AI,..."
5,5,AI Health and Biotech - Shopify,9,Germany,ICT,"Shopify, Ant Group, SoftBank Group, UiPath, Owkin"
6,6,Foundation Model Ecosystem - Google,11,United States,AI model,"Google, Alphabet Inc., Amazon, ByteDance, Deep..."
7,7,AI Chip Infrastructure,17,United States,Hardware,"Nvidia, Broadcom, AWS, AMD, Cisco"
8,8,Foundation Model Ecosystem - Y combinator,10,Unknown,AI model,"Y combinator, Scale AI, Tempus AI, Semrush, Ox..."


## 10. Final Method Notes

This notebook keeps one analysis path for community detection:
- Build a weighted k-NN graph in 3D t-SNE space.
- Detect communities with Louvain on that graph.
- Refine tiny groups with local reassignment for cleaner clusters.
- Render the final map in 2D for interpretation.
- Color blobs by mean z (yellow -> green).
- Export assignments and short community labels for audit.

No alternative clustering workflow is kept in this narrative.

In [22]:
print("-- Export summary --")
export_paths = [raw_path, long_path, agg_path, cooc_path, coords_path, fig_html]

for maybe_var in ["communities_path", "fig_louvain_html", "interpretation_path"]:
    if maybe_var in globals():
        export_paths.append(globals()[maybe_var])

for p in [
    EXPORTS_DIR / "communities_kmeans_2d.csv",
    EXPORTS_DIR / "community_labels_short.csv",
    EXPORTS_DIR / "competition_map_2d_kmeans.html",
]:
    if p not in export_paths:
        export_paths.append(p)

seen = set()
ordered_paths = []
for p in export_paths:
    p = Path(p)
    key = str(p)
    if key not in seen:
        seen.add(key)
        ordered_paths.append(p)

for p in ordered_paths:
    if p.exists():
        size_kb = p.stat().st_size / 1024
        print(f"  {p.name:<42} {size_kb:6.1f} KB")
    else:
        print(f"  {p.name:<42} {'MISSING':>6}")

-- Export summary --
  competitors_raw.csv                          11.0 KB
  competitors_long.csv                         32.4 KB
  competitors_aggregated.csv                   12.4 KB
  cooccurrence_matrix.csv                     140.2 KB
  coords_3d.csv                                 7.7 KB
  competition_map_2d.html                    4784.9 KB
  communities_kmeans_2d.csv                     6.8 KB
  competition_map_2d_kmeans.html             4797.8 KB
  community_labels_short.csv                    1.1 KB
